In [1]:
!pip install geopandas

In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import geopandas as gpd
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt
import pickle
import os
import gc
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.preprocessing import LabelEncoder

# ==========================================
# 0. 設定 & ディレクトリ作成
# ==========================================
# モデル保存先を作成
model_dir = '/work/Real_estate/models/'
os.makedirs(model_dir, exist_ok=True)

# GIS設定
shp_file_path = '/work/Real_estate/datas/L01-25.shp'
target_price_col = 'L01_008'

# ==========================================
# 1. データ読み込み & 前処理 (GIS結合含む)
# ==========================================
print("Reading Data...")
train = pd.read_csv('/work/Real_estate/train/train.csv')
test = pd.read_csv('/work/Real_estate/test/test.csv')
sample_sub = pd.read_csv('/work/Real_estate/sample_submit.csv', header=None)

# 外れ値除去
train = train[(train['unit_area'] > 10) & (train['unit_area'] < 500)].copy()

# 共通処理関数
def preprocess(df):
    # 築年数 (月単位)
    def calculate_age(row):
        try:
            if pd.isna(row['year_built']) or pd.isna(row['target_ym']): return np.nan
            t_y, t_m = divmod(int(row['target_ym']), 100)
            b_y, b_m = divmod(int(row['year_built']), 100)
            return (t_y - b_y) * 12 + (t_m - b_m)
        except: return np.nan
    df['building_age'] = df.apply(calculate_age, axis=1)
    return df

train = preprocess(train)
test = preprocess(test)

# カテゴリエンコード
cols_cat = ['addr1_1', 'addr1_2']
for c in cols_cat:
    le = LabelEncoder()
    all_vals = pd.concat([train[c], test[c]]).astype(str).unique()
    le.fit(all_vals)
    train[f'{c}_code'] = le.transform(train[c].astype(str))
    test[f'{c}_code'] = le.transform(test[c].astype(str))

# --- GISデータ結合 ---
print(f"Loading GIS: {shp_file_path}")
try:
    gdf = gpd.read_file(shp_file_path)
    land_prices = gdf[target_price_col].astype(float).values
    land_coords = np.array(list(zip(gdf.geometry.x, gdf.geometry.y)))
    
    del gdf; gc.collect()
    
    print("Mapping Nearest Land Price...")
    tree = cKDTree(land_coords)
    
    def add_gis(df):
        dists, idxs = tree.query(df[['lon', 'lat']].values, k=1)
        df['nearest_land_price'] = land_prices[idxs]
        return df

    train = add_gis(train)
    test = add_gis(test)
    
except Exception as e:
    print(f"🚨 GIS Error: {e}")
    train['nearest_land_price'] = 0
    test['nearest_land_price'] = 0

# ==========================================
# 2. 学習関数の定義 (Regression用に修正)
# ==========================================
def train_lgb(input_x,
              input_y,
              input_id,
              params,
              model_prefix="model", # 保存ファイル名の識別子
              n_splits=5,
             ):
    
    # input_x のインデックスをリセットして扱いやすくする
    input_x = input_x.reset_index(drop=True)
    input_y = input_y.reset_index(drop=True)
    input_id = input_id.reset_index(drop=True)
    
    train_oof = np.zeros(len(input_x))
    metrics = []
    imp = pd.DataFrame()

    # cross-validation (回帰なのでKFoldを使用)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=123)
    
    for nfold, (idx_tr, idx_va) in enumerate(kf.split(input_x, input_y)):
        print("-" * 20, f"Fold {nfold}", "-" * 20)
        
        # make dataset
        x_tr, y_tr = input_x.loc[idx_tr, :], input_y[idx_tr]
        x_va, y_va = input_x.loc[idx_va, :], input_y[idx_va]
        
        # 【重要】目的変数を対数変換 (MAPE対策)
        y_tr_log = np.log1p(y_tr)
        y_va_log = np.log1p(y_va)
        
        # train (Regressorを使用)
        model = lgb.LGBMRegressor(**params)
        
        # 新しいバージョンのLightGBMに対応したCallback記述
        callbacks = [
            lgb.early_stopping(stopping_rounds=100, verbose=True),
            lgb.log_evaluation(period=100)
        ]
        
        model.fit(x_tr,
                  y_tr_log,
                  eval_set=[(x_tr, y_tr_log), (x_va, y_va_log)],
                  eval_metric='rmse',
                  callbacks=callbacks
                 )
        
        # モデル保存
        fname_lgb = os.path.join(model_dir, f"{model_prefix}_fold{nfold}.pickle")
        with open(fname_lgb, "wb") as f:
            pickle.dump(model, f, protocol=4)
        
        # evaluate (対数を元に戻して評価)
        y_tr_pred_log = model.predict(x_tr)
        y_va_pred_log = model.predict(x_va)
        
        y_tr_pred = np.expm1(y_tr_pred_log)
        y_va_pred = np.expm1(y_va_pred_log)
        
        # Metric: MAPE
        metric_tr = mean_absolute_percentage_error(y_tr, y_tr_pred)
        metric_va = mean_absolute_percentage_error(y_va, y_va_pred)
        
        metrics.append([nfold, metric_tr, metric_va])
        print(f"[MAPE] tr:{metric_tr:.4f}, va:{metric_va:.4f}")
        
        # oof
        train_oof[idx_va] = y_va_pred
        
        # imp
        _imp = pd.DataFrame({"col":input_x.columns, "imp":model.feature_importances_, "nfold":nfold})
        imp = pd.concat([imp, _imp])
      
    print("-" * 20, "result", "-" * 20)
    # metric
    metrics = np.array(metrics)
    # print(metrics)
    print("[cv] tr:{:.4f}+-{:.4f}, va:{:.4f}+-{:.4f}".format(
        metrics[:,1].mean(), metrics[:,1].std(),
        metrics[:,2].mean(), metrics[:,2].std(),
    ))
    
    # OOF全体でのスコア
    oof_score = mean_absolute_percentage_error(input_y, train_oof)
    print(f"[oof] {oof_score:.4f}")
    
    # oof dataframe check
    train_oof_df = pd.concat([
        input_id,
        pd.DataFrame({"pred": train_oof})
    ], axis=1)
    
    # importance
    imp = imp.groupby("col")["imp"].agg(["mean", "std"]).reset_index(drop=False)
    imp.columns = ["col", "imp", "imp_std"]
    
    return train_oof_df, imp, metrics

# ==========================================
# 3. 実行 (マンション・戸建て別)
# ==========================================
# パラメータ設定
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'n_estimators': 10000,
    'random_state': 42,
    'verbosity': -1
}

# 特徴量定義
features_m = [
    'unit_area', 'building_age', 'walk_distance1', 'bus_time1',
    'parking_money', 'floor_count', 'room_floor',
    'super_distance', 'lon', 'lat', 'addr1_1_code', 'addr1_2_code',
    'nearest_land_price'
]

features_h = [
    'unit_area', 'building_age', 'walk_distance1', 'bus_time1',
    'super_distance', 'school_ele_distance', 'lon', 'lat',
    'addr1_1_code', 'addr1_2_code',
    'nearest_land_price'
]

# --- ① マンション学習 ---
print("\n===== Training Mansion Model =====")
df_m = train[train['bukken_type'] == 1302]
oof_m, imp_m, met_m = train_lgb(
    input_x=df_m[features_m],
    input_y=df_m['money_room'],
    input_id=df_m[['target_ym']], # ID代わり
    params=params,
    model_prefix="mansion_lgb"
)

# --- ② 戸建て学習 ---
print("\n===== Training House Model =====")
df_h = train[train['bukken_type'] == 1202]
oof_h, imp_h, met_h = train_lgb(
    input_x=df_h[features_h],
    input_y=df_h['money_room'],
    input_id=df_h[['target_ym']], # ID代わり
    params=params,
    model_prefix="house_lgb"
)

# ==========================================
# 4. テスト予測 & 提出ファイル作成
# ==========================================
print("\n===== Generating Submission =====")

def predict_test(test_df, features, model_prefix, n_splits=5):
    preds = np.zeros(len(test_df))
    
    for nfold in range(n_splits):
        fname = os.path.join(model_dir, f"{model_prefix}_fold{nfold}.pickle")
        with open(fname, "rb") as f:
            model = pickle.load(f)
        
        # 対数で予測されるので戻す
        pred_log = model.predict(test_df[features])
        preds += np.expm1(pred_log) / n_splits
        
    return preds

# マンション予測
idx_m = test['bukken_type'] == 1302
if idx_m.sum() > 0:
    test.loc[idx_m, 'prediction'] = predict_test(test[idx_m], features_m, "mansion_lgb")

# 戸建て予測
idx_h = test['bukken_type'] == 1202
if idx_h.sum() > 0:
    test.loc[idx_h, 'prediction'] = predict_test(test[idx_h], features_h, "house_lgb")

# 保存
sample_sub.iloc[:, 1] = test['prediction'].values
sample_sub.to_csv('submission.csv', index=False, header=False)

print("✅ Submission file created: submission.csv")
print(f"Models saved in: {model_dir}")

Reading Data...


<ipython-input-2-0436fd1c074c>:29: DtypeWarning: Columns (63) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv('/work/Real_estate/train/train.csv')
<ipython-input-2-0436fd1c074c>:30: DtypeWarning: Columns (46,55,56,63,146) have mixed types. Specify dtype option on import or set low_memory=False.
  test = pd.read_csv('/work/Real_estate/test/test.csv')
/usr/local/lib/python3.10/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater
  return op(a, b)
/usr/local/lib/python3.10/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less
  return op(a, b)


Loading GIS: /work/Real_estate/datas/L01-25.shp
Mapping Nearest Land Price...

===== Training Mansion Model =====
-------------------- Fold 0 --------------------
Training until validation scores don't improve for 100 rounds
[100]	training's rmse: 0.23035	valid_1's rmse: 0.232043
[200]	training's rmse: 0.215542	valid_1's rmse: 0.219381
[300]	training's rmse: 0.208368	valid_1's rmse: 0.213821
[400]	training's rmse: 0.20353	valid_1's rmse: 0.21041
[500]	training's rmse: 0.19973	valid_1's rmse: 0.208102
[600]	training's rmse: 0.196415	valid_1's rmse: 0.206169
[700]	training's rmse: 0.193544	valid_1's rmse: 0.204684
[800]	training's rmse: 0.191026	valid_1's rmse: 0.203419
[900]	training's rmse: 0.18873	valid_1's rmse: 0.202296
[1000]	training's rmse: 0.186566	valid_1's rmse: 0.201289
[1100]	training's rmse: 0.184534	valid_1's rmse: 0.200324
[1200]	training's rmse: 0.182649	valid_1's rmse: 0.199532
[1300]	training's rmse: 0.180871	valid_1's rmse: 0.198816
[1400]	training's rmse: 0.179169	va

<ipython-input-2-0436fd1c074c>:268: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[14969632.84222364 27986564.67703941 11061276.93885564 ...
 16067123.97814825 18030494.92939517 14082506.21109078]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  sample_sub.iloc[:, 1] = test['prediction'].values


✅ Submission file created: submission.csv
Models saved in: /work/Real_estate/models/
